# TS-models robustness: refit each model's HPO-selected best config with 2 new seeds

Only `gru`/`gru_mimo`/`nbeats`/`tft` actually vary with seed; the rest are deterministic (std dev = 0 expected).

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))

from ts_comparison import load_splits, STAT_FORECASTERS
from extra_forecasters import EXTRA_FORECASTERS
from prophet_baseline import _fit_prophet
from error_direction import _best_params_for, STAT_MODELS, ML_MODELS, DARTS_MODELS
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness" / "ts_models"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
SERIES = ["concurrent_cases", "throughput_time"]

NEW_SEEDS = [43, 44]

ALL_TS_MODELS = {**STAT_FORECASTERS, **EXTRA_FORECASTERS}
try:
    from time_series_prediction import forecast_nbeats, forecast_tft
    ALL_TS_MODELS["nbeats"] = forecast_nbeats
    ALL_TS_MODELS["tft"] = forecast_tft
except Exception as e:
    print(f"  [warn] darts forecasters unavailable ({e}) -- skipping nbeats/tft")

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, "
     f"{len(ALL_TS_MODELS)} ts models (+ prophet, val_mean, naive handled separately), seeds={NEW_SEEDS}")

In [ ]:
def run_ts_models_robustness_one(dataset: str, is_real: bool, seed: int):
    """One (dataset, seed) robustness run across every ts model + prophet. Skips if metrics already exist."""
    sub = "ssd" if is_real else "synthetic"
    trim = "ssd" if is_real else "none"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        _, cc, tt, _, _, _ = load_data_for_ssd(xes_path)
        splits = {"concurrent_cases": cc, "throughput_time": tt}
    else:
        split = load_splits(dataset, "none", is_real=False)
        splits = {"concurrent_cases": split["cc"], "throughput_time": split["tt"]}

    rows = []
    for series_name in SERIES:
        s = splits[series_name]
        train_val = pd.concat([s["train"], s["val"]])
        test_idx = s["test"].index
        actual = s["test"].to_numpy()
        if len(actual) == 0:
            continue

        for model_name, forecaster in ALL_TS_MODELS.items():
            params = _best_params_for(model_name, dataset, trim, is_real, series_name)
            params = dict(params) if params else {}
            params["seed"] = seed
            try:
                pred = forecaster(train_val, len(test_idx), params)
            except Exception as e:
                print(f"    [FAILED] {model_name}/{series_name}: {e}")
                continue
            rows.append(dict(dataset=dataset, series=series_name, model=model_name, seed=seed,
                             mae=mean_absolute_error(actual, pred), mse=mean_squared_error(actual, pred)))

        prophet_params = _best_params_for("prophet", dataset, trim, is_real, series_name)
        try:
            pred_p = (_fit_prophet(train_val, test_idx, prophet_params or {})
                     .reindex(test_idx).ffill().bfill().fillna(0).to_numpy())
            rows.append(dict(dataset=dataset, series=series_name, model="prophet", seed=seed,
                             mae=mean_absolute_error(actual, pred_p), mse=mean_squared_error(actual, pred_p)))
        except Exception as e:
            print(f"    [FAILED] prophet/{series_name}: {e}")

    if not rows:
        print(f"  [skip] {dataset}/seed_{seed}: no rows produced (test set empty?)")
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(metrics_path, index=False)
    print(f"  [done] {dataset}/seed_{seed}: {len(rows)} (model, series) rows")

## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_ts_models_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_ts_models_robustness_one(name, is_real=True, seed=seed)